# 04 — Système multi-agents

Ce notebook présente le système multi-agents (chercheur, rédacteur, vérificateur), sa mise au point sur le jeu **dev** et sa comparaison avec le RAG simple sur ce même jeu.

Quatre évaluations sont utilisées, toutes avec `qwen3:4b-instruct`, le même index et au plus 5 passages par réponse :

| Évaluation | Système | Rôle |
| --- | --- | --- |
| `simple-dev` | RAG simple | référence de l'étape 3 (juge v1) |
| `agents-v1-dev` | agents, première version | diagnostic |
| `agents-v2-dev` | agents après corrections | version retenue |
| `simple-v2-dev` | RAG simple, même juge et même règle de refus que `agents-v2-dev` | comparaison équitable |

Le jeu **test** reste inutilisé : la comparaison finale est faite à l'étape 5.

In [1]:
import pandas as pd

from assistant_rag.agents.graph import Researcher, Verifier, Writer, build_graph
from assistant_rag.config import RESULTS_DIR, Settings
from assistant_rag.evaluation.judge import VERDICT_SCORES
from assistant_rag.evaluation.metrics import score_records, summarize
from assistant_rag.evaluation.runner import load_records

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 2)

runs = ["simple-dev", "agents-v1-dev", "simple-v2-dev", "agents-v2-dev"]
records = {run: load_records(RESULTS_DIR / run / "answers.jsonl") for run in runs}
answers = {run: pd.DataFrame(recs).set_index("id") for run, recs in records.items()}
scores = {run: score_records(recs).set_index("id") for run, recs in records.items()}

## 1. Architecture

Le graphe LangGraph relie les trois agents. Le vérificateur renvoie la réponse au rédacteur si une affirmation n'est pas justifiée par l'extrait cité, ou au chercheur si un aspect de la question n'est couvert par aucun passage. Après 2 corrections sans validation, la première réponse est conservée et marquée comme non vérifiée.

In [2]:
graph = build_graph(Researcher(None, None, Settings()), Writer(None), Verifier(None), max_revisions=2)
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	chercheur(chercheur)
	redacteur(redacteur)
	verificateur(verificateur)
	__end__([<p>__end__</p>]):::last
	__start__ --> chercheur;
	chercheur --> redacteur;
	redacteur --> verificateur;
	verificateur -.-> __end__;
	verificateur -.-> chercheur;
	verificateur -.-> redacteur;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. Première version : diagnostic

In [3]:
summarize(records["agents-v1-dev"])

,Questions,Hit@k,Rappel@k,MRR,Précision des citations,Rappel des citations,Exactitude,Refus à tort,Refus correct,Latence moyenne (s),Appels au LLM
Factuelles,24,1.0,1.0,0.94,0.72,0.79,0.71,0.21,NaN,17.05,3.38
Multi-articles,8,1.0,1.0,0.94,0.80,0.94,0.62,0.00,NaN,26.02,4.75
Hors corpus,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,3.32,2.00
Ensemble,40,1.0,1.0,0.94,0.74,0.83,0.69,0.16,1.0,16.10,3.38


Par rapport au RAG simple, les questions multi-articles retrouvent tous leurs articles (rappel 100 % contre 81 %), mais la précision des citations chute à 74 % et 5 questions factuelles sont refusées à tort. Le nombre de sous-questions produites par le chercheur explique ce résultat :

In [4]:
def sub_question_counts(run):
    frame = answers[run]
    counts = frame["trace"].apply(lambda trace: len(trace[0]["requetes"]))
    return pd.crosstab(frame["type"], counts, colnames=["recherches lancées"])


sub_question_counts("agents-v1-dev")

recherches lancées,1,2,3,4
type,,,,
factuelle,2,3,15,4
hors_corpus,7,0,0,1
multi_articles,0,0,6,2


22 questions factuelles sur 24 sont découpées alors qu'elles ne portent que sur un point. Le chercheur ajoute souvent une sous-question sur l'autre règlement, absente de la question :

In [5]:
for step in answers["agents-v1-dev"].loc["rgpd-06", "trace"]:
    details = {key: value for key, value in step.items() if key != "agent"}
    print(f"{step['agent']} : {str(details)[:600]}\n")

chercheur : {'requetes': ['Quels sont les principes relatifs au traitement des données à caractère personnel ?', 'Quels sont les principes du traitement des données à caractère personnel dans le RGPD ?', "Quels sont les principes du traitement des données à caractère personnel dans le règlement européen sur l'IA ?"], 'passages': ['[1] RGPD art. 5', '[2] AI Act art. 59', '[3] RGPD art. 5', '[4] RGPD art. 1', '[5] AI Act art. 2']}

rédacteur : {'reponse': "Les principes relatifs au traitement des données à caractère personnel dans le RGPD sont : licité, loyale et transparente ; collectées pour des finalités déterminées, explicites et légitimes, sans traitement ultérieur incompatible avec ces finalités ; adéquates, pertinentes et limitées à ce qui est nécessaire ; exactes et tenues à jour ; conservées pendant une durée nécessaire ou plus longue dans des cas spécifiques comme les fins archivistiques, scientifiques ou statistiques avec mesures appropriées ; traitées de façon à garantir une 

Le rédacteur répond alors « je ne trouve pas » pour l'aspect inventé, et la réponse entière est comptée comme un refus. Les traces montrent un second défaut : le vérificateur conteste des affirmations exactes, et les corrections dégradent parfois une bonne réponse.

**Corrections apportées (version `agents-v2`) :**

- le chercheur ne découpe la question que si elle demande explicitement plusieurs informations et n'ajoute jamais un règlement absent de la question (deux exemples dans son prompt) ;
- le vérificateur ne signale que des erreurs certaines : chiffre absent ou différent de l'extrait cité, affirmation attribuée au mauvais extrait, aspect non couvert ;
- si les corrections ne convergent pas, la première réponse est conservée et marquée comme non vérifiée ;
- une réponse qui traite une partie de la question avec des citations n'est plus comptée comme un refus total (règle appliquée aux deux systèmes).

## 3. Version retenue

In [6]:
sub_question_counts("agents-v2-dev")

recherches lancées,1,3
type,,
factuelle,22,2
hors_corpus,8,0
multi_articles,2,6


Le découpage ne concerne plus que les questions qui en ont besoin : 6 questions multi-articles sur 8 et 2 questions factuelles qui portent sur deux acteurs.

In [7]:
summarize(records["agents-v2-dev"])

,Questions,Hit@k,Rappel@k,MRR,Précision des citations,Rappel des citations,Exactitude,Refus à tort,Refus correct,Latence moyenne (s),Appels au LLM
Factuelles,24,1.0,1.00,0.94,0.96,1.00,0.88,0.0,NaN,11.40,3.5
Multi-articles,8,1.0,0.94,0.94,0.90,0.94,0.75,0.0,NaN,20.80,4.0
Hors corpus,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,2.62,2.0
Ensemble,40,1.0,0.98,0.94,0.94,0.98,0.84,0.0,1.0,11.52,3.3


## 4. Comparaison avec le RAG simple sur le jeu dev

Les deux systèmes sont évalués dans les mêmes conditions (`simple-v2-dev` et `agents-v2-dev`).

In [8]:
key_metrics = ["Rappel@k", "Précision des citations", "Rappel des citations", "Exactitude", "Refus correct",
               "Latence moyenne (s)", "Appels au LLM"]
comparison = pd.concat(
    {"RAG simple": summarize(records["simple-v2-dev"]), "Agents": summarize(records["agents-v2-dev"])}, axis=1
)
comparison.swaplevel(axis=1)[key_metrics]

Rappel@k        Précision des citations         \
               RAG simple Agents              RAG simple Agents   
Factuelles           1.00   1.00                    0.94   0.96   
Multi-articles       0.81   0.94                    0.88   0.90   
Hors corpus           NaN    NaN                     NaN    NaN   
Ensemble             0.95   0.98                    0.92   0.94   

               Rappel des citations        Exactitude        Refus correct  \
                         RAG simple Agents RAG simple Agents    RAG simple   
Factuelles                     0.96   1.00       0.90   0.88           NaN   
Multi-articles                 0.81   0.94       0.69   0.75           NaN   
Hors corpus                     NaN    NaN        NaN    NaN           1.0   
Ensemble                       0.92   0.98       0.84   0.84           1.0   

                      Latence moyenne (s)        Appels au LLM         
               Agents          RAG simple Agents    RAG simple Agents  
Factuelles        NaN                7.71  11.40           1.0    3.5  
Multi-articles    NaN                6.10  20.80           1.0    4.0  
Hors corpus       1.0                2.49   2.62           1.0    2.0  
Ensemble          1.0                6.35  11.52           1.0    3.3

La recherche par sous-question améliore la couverture des questions multi-articles : le rappel des articles retrouvés passe de 81 % à 94 % et celui des articles cités de 81 % à 94 %. Les questions factuelles ont des résultats équivalents.

### Relecture humaine

Les 32 réponses notées par le juge ont été relues pour chacun des deux systèmes (`revue_humaine.csv`).

In [9]:
def human_scores(run):
    review = pd.read_csv(RESULTS_DIR / run / "revue_humaine.csv").set_index("id")
    answerable = scores[run][scores[run]["type"] != "hors_corpus"]
    human = answerable["correctness"].copy()
    human.loc[review.index] = review["verdict_humain"].map(VERDICT_SCORES)
    table = human.groupby(answerable["type"]).mean()
    table["ensemble"] = human.mean()
    agreement = (review["verdict_juge"] == review["verdict_humain"]).mean()
    return table, agreement


rows, agreements = {}, {}
for label, run in [("RAG simple", "simple-v2-dev"), ("Agents", "agents-v2-dev")]:
    rows[label], agreements[label] = human_scores(run)
exactness = pd.DataFrame(rows)
exactness.loc["accord juge / relecture"] = pd.Series(agreements)
exactness

,RAG simple,Agents
type,,
factuelle,0.94,0.94
multi_articles,0.75,0.81
ensemble,0.89,0.91
accord juge / relecture,0.81,0.84


In [10]:
reviews = {
    label: pd.read_csv(RESULTS_DIR / run / "revue_humaine.csv").set_index("id")
    for label, run in [("RAG simple", "simple-v2-dev"), ("Agents", "agents-v2-dev")]
}
verdicts = pd.DataFrame({label: review["verdict_humain"] for label, review in reviews.items()})
changed = verdicts[verdicts["RAG simple"] != verdicts["Agents"]]
changed.join(reviews["Agents"]["commentaire"].rename("commentaire (agents)"))

,RAG simple,Agents,commentaire (agents)
id,,,
rgpd-11,correct,incorrect,"Montant de 20 000 000 EUR au lieu de 10 000 000 EUR ; la version corrigée par le rédacteur était juste, mais la première réponse a été c..."
aiact-06,incorrect,correct,NaN
multi-11,incorrect,partiel,Règle du RGPD exacte grâce à l'article 9 retrouvé ; une seule des interdictions biométriques de l'AI Act est citée.


Les deux systèmes ne se trompent pas sur les mêmes questions :

- **multi-11** : le RAG simple invente une règle du RGPD faute d'avoir retrouvé l'article 9 ; le chercheur le retrouve grâce à la sous-question dédiée ;
- **aiact-06** : le RAG simple donne le montant de l'article 100 (1 500 000 EUR) ; les agents donnent le bon montant de l'article 99 ;
- **rgpd-11** : les agents donnent un montant faux alors que le RAG simple répond juste.

## 5. Traces commentées

In [11]:
def show_trace(run, question_id, width=500):
    print(answers[run].loc[question_id, "question"], "\n")
    for step in answers[run].loc[question_id, "trace"]:
        details = {key: value for key, value in step.items() if key != "agent"}
        print(f"{step['agent']} : {str(details)[:width]}\n")


show_trace("agents-v2-dev", "multi-11")

Comment le RGPD protège-t-il les données biométriques et quels usages de la biométrie l'AI Act interdit-il ? 

chercheur : {'requetes': ["Comment le RGPD protège-t-il les données biométriques et quels usages de la biométrie l'AI Act interdit-il ?", 'Comment le RGPD protège-t-il les données biométriques ?', "Quels usages de la biométrie l'AI Act interdit-il ?"], 'passages': ['[1] AI Act art. 5', '[2] RGPD art. 1', '[3] AI Act art. 5', '[4] RGPD art. 25', '[5] RGPD art. 9']}

rédacteur : {'reponse': "Le RGPD protège les données biométriques en interdisant leur traitement à des fins d'identification unique d'une personne physique, sauf dans des cas spécifiques autorisés par des conditions telles que le consentement explicite ou la nécessité pour l'exécution de droits professionnels ou sociaux, ou la sauvegarde des intérêts vitaux [5].  \n\nL'AI Act interdit l'utilisation de systèmes d'identification biométrique à distance en temps réel dans des espaces accessibles au public à des

vérific

Le chercheur lance une recherche par règlement et retrouve l'article 9 du RGPD, absent des passages du RAG simple. Le vérificateur conteste ensuite des affirmations exactes ; les corrections ne convergent pas et la première réponse, correcte sur le RGPD, est conservée.

In [12]:
show_trace("agents-v2-dev", "rgpd-11")

Quel est le montant maximal de l'amende en cas de violation des obligations incombant au responsable du traitement et au sous-traitant ? 

chercheur : {'requetes': ["Quel est le montant maximal de l'amende en cas de violation des obligations incombant au responsable du traitement et au sous-traitant ?", "Quel est le montant maximal de l'amende en cas de violation des obligations incombant au responsable du traitement ?", "Quel est le montant maximal de l'amende en cas de violation des obligations incombant au sous-traitant ?"], 'passages': ['[1] RGPD art. 83', '[2] AI Act art. 99', '[3] RGPD art. 83', '[4] RGPD art. 82', '[5] AI Act art. 100']

rédacteur : {'reponse': "Le montant maximal de l'amende en cas de violation des obligations incombant au responsable du traitement est de 20 000 000 EUR ou, dans le cas d'une entreprise, jusqu'à 4 % du chiffre d'affaires annuel mondial total de l'exercice précédent, le montant le plus élevé étant retenu [1].  \nLe montant maximal de l'amende en 

Cas inverse : la troisième réponse du rédacteur donnait le bon montant (10 000 000 EUR), mais le vérificateur ne l'a pas validée et la première réponse, fausse, a été conservée.

## 6. Rôle du vérificateur

Dans la version retenue, le vérificateur valide 27 des 32 réponses dès la première rédaction. Pour les 5 autres, ses corrections ne convergent jamais : il n'améliore donc aucune réponse finale. Il reste utile comme **signal de doute** : 3 des 5 réponses non validées sont fausses ou incomplètes, contre 2 sur 27 parmi les réponses validées.

In [13]:
review = reviews["Agents"]
flags = answers["agents-v2-dev"].loc[review.index, "verified"].map({True: "validée", False: "non validée"})
pd.crosstab(flags, review["verdict_humain"], margins=True, margins_name="total")

verdict_humain,correct,incorrect,partiel,total
verified,,,,
non validée,2,1,2,5
validée,25,0,2,27
total,27,1,4,32


Avec un modèle de 4 milliards de paramètres, le vérificateur produit encore trop d'objections infondées pour corriger les réponses de façon fiable. Deux pistes seront mesurées à l'étape 6 : un vérificateur utilisant un modèle plus grand, et une version sans vérificateur pour isoler l'apport du chercheur.

## 7. Coût

In [14]:
cost = pd.DataFrame(
    {
        label: {
            "latence médiane (s)": scores[run]["latency"].median(),
            "latence moyenne (s)": scores[run]["latency"].mean(),
            "appels au LLM par question": scores[run]["llm_calls"].mean(),
        }
        for label, run in [("RAG simple", "simple-v2-dev"), ("Agents", "agents-v2-dev")]
    }
)
cost

,RAG simple,Agents
latence médiane (s),4.38,7.04
latence moyenne (s),6.35,11.52
appels au LLM par question,1.00,3.30


## 8. Conclusions

**Ce que les agents apportent.** La décomposition de la question et la recherche par sous-question corrigent le principal défaut du RAG simple : sur les questions multi-articles, le rappel des articles retrouvés passe de 81 % à 94 %, et l'exactitude après relecture de 0,75 à 0,81. Les deux erreurs les plus graves du RAG simple sur le jeu dev (montant de l'article 100 au lieu de l'article 99, règle du RGPD inventée) disparaissent.

**Ce qu'ils n'apportent pas.** Sur l'ensemble des 32 questions avec réponse, l'exactitude relue est de 0,91 contre 0,89 : l'écart correspond à une demi-question et n'est pas significatif sur un jeu de cette taille. Les questions factuelles ont la même exactitude (0,94). Le vérificateur, limité par la taille du modèle, ne corrige aucune réponse ; il sert seulement à signaler les réponses douteuses.

**Coût.** Les agents font en moyenne 3,3 appels au LLM par question et multiplient le temps de réponse par 1,6 (médiane) à 1,8 (moyenne).

**Itération.** La première version dégradait les résultats (5 refus à tort, citations moins précises). L'analyse des traces a permis d'identifier les deux causes et de les corriger sur le jeu dev, sans utiliser le jeu test.

**Suite.** L'étape 5 compare les deux systèmes sur le jeu test, réservé jusqu'ici, pour confirmer ou non ces tendances sur des questions qui n'ont servi à aucun réglage.